# CIF -> master pattern -> detector

Takes a CIF file, simulates a dynamical EBSD master pattern from it, displays the
result with anyplotlib, and projects it onto a detector at chosen orientations.

Runs on GPU if one is available, CPU otherwise. The master-pattern step is the only
slow cell (~10 s on a Titan X at the settings below; minutes on CPU).

In [1]:
import time

import numpy as np
import torch
from orix.quaternion import Rotation

import de_ebsd as de

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"de-ebsd {de.__version__} | torch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  {torch.cuda.get_device_name(0)}")

de-ebsd 0.1.0 | torch 2.6.0+cu124 | device: cuda
  NVIDIA TITAN X (Pascal)


## 1. Load the crystal from a CIF

`Crystal.from_cif` reads the **asymmetric unit** only -- `Crystal` applies the
space-group symmetry itself, so handing it the expanded cell that
`diffpy.structure.loadStructure` returns would count every atom twice.

It also converts CIF Angstroms to nm, turns a `U_iso` column into the Debye-Waller
factor `B = 8 pi^2 U_iso` (nm^2), and reduces ionic labels to the element
(`Si4+` -> `Si`), since the scattering tables are per element.

In [3]:
CIF_PATH = "Si.cif"  # <- point this at your own CIF

crystal = de.Crystal.from_cif(CIF_PATH)
crystal

Crystal(a=0.5431 nm, b=0.5431, c=0.5431, alpha=90, beta=90, gamma=90, space_group=227, atoms=[Si(8x)])

In [4]:
print(crystal)
print(f"space group : {crystal.space_group}  (origin choice {crystal.origin})")
print(f"atoms/cell  : {sum(len(p) for p in crystal.positions)}")
print(f"density     : {crystal.density:.3f} g/cm^3")
print(f"centering   : {crystal.centering}")
print("asymmetric unit:")
for site in crystal.sites:
    x, y, z = site.position
    print(
        f"  {site.element:<2}  ({x:.4f}, {y:.4f}, {z:.4f})"
        f"  occ={site.occupancy:.2f}  B={site.debye_waller:.5f} nm^2"
    )

Crystal(a=0.5431 nm, b=0.5431, c=0.5431, alpha=90, beta=90, gamma=90, space_group=227, atoms=[Si(8x)])
space group : 227  (origin choice 2)
atoms/cell  : 8
density     : 2.329 g/cm^3
centering   : F
asymmetric unit:
  Si  (0.1250, 0.1250, 0.1250)  occ=1.00  B=0.00497 nm^2


> **Check the origin choice.** For the 24 space groups the International Tables list
> with two origins, the choice moves the atoms and therefore changes the master
> pattern -- and a wrong one fails quietly, producing a plausible pattern that does not
> match reality. Most CIFs do not record it, so `from_cif` defaults to **choice 2**
> (origin on an inversion centre; what EMsoft and kikuchipy use). Override with
> `Crystal.from_cif(path, origin=1)` if your file uses the other setting.

In [6]:
crystal.show()

Figure(1x1, panels=1, size=480x420)

In [ ]:
if crystal.has_two_origins:
    print(
        f"space group {crystal.space_group} has two origin choices; "
        f"using {crystal.origin}. Pass origin=... to from_cif to override."
    )
else:
    print(f"space group {crystal.space_group} has a single origin -- nothing to choose.")

In [ ]:
crystal.show()

## 2. Set up the simulation

`backend="bloch"` is the dynamical solver and what you want for a real master pattern.
The default, `"kinematical"`, is a geometry-only stand-in that gets band *positions*
right but not their intensities -- useful for validation, not for indexing.

`npx` and `dmin` are the accuracy/cost knobs: `npx` sets the Lambert grid
(`(2*npx+1)^2` per hemisphere) and `dmin` the smallest interplanar spacing still
included as a reflection.

In [7]:
microscope = de.Microscope(energy=20e3, sample_tilt=70.0)

simulator = de.MasterPatternSimulator(
    crystal,
    microscope,
    npx=100,
    dmin=0.05,
    backend="bloch",
)
simulator

MasterPatternSimulator(Crystal(a=0.5431 nm, b=0.5431, c=0.5431, alpha=90, beta=90, gamma=90, space_group=227, atoms=[Si(8x)]), Microscope(energy=20000 eV, sample_tilt=70 deg, azimuthal=0 deg), npx=100, dmin=0.1 nm, backend='bloch')

Each stage of the chain is an inspectable object before anything is computed. The
structure factors are the expensive setup step and are cached on the simulator.

In [9]:
microscope.show()

In [ ]:
structure_factors = simulator.structure_factors()
print(structure_factors)
structure_factors.show()

## 3. Compute the master pattern

The Bloch-wave backend integrates over exit depth, so it needs Monte Carlo
energy/depth statistics. None were supplied here, so `compute` runs de_ebsd's own
Monte Carlo at the sample tilt -- no EMsoft installation required.

In [ ]:
t0 = time.perf_counter()
master = simulator.compute(device=DEVICE)
print(f"computed in {time.perf_counter() - t0:.1f} s")
print(f"data  {tuple(master.data.shape)}  = (n_energy, hemisphere, n, n)  on {master.device}")
print(f"energies: {master.energies_kev} keV")

## 4. Display it

Every `.show()` renders through `de_ebsd.viz` (anyplotlib), returning a figure that
is interactive in Jupyter and exportable with `figure.save_html(path)`.

`"lambert"` is the square-Lambert storage grid itself -- both hemispheres, no
resampling, so it is the honest view of the underlying data.

In [ ]:
master.show(projection="lambert")

`"stereographic"` resamples to the equal-angle projection crystallographers read.

In [ ]:
master.show(projection="stereographic")

`"sphere"` textures the pattern onto an interactive 3-D sphere -- drag to rotate.
This is what the master pattern actually *is*: a function on the sphere.

In [ ]:
master.show(projection="sphere")

## 5. Project onto a detector

`EBSDDetector` uses the Bruker pattern-center convention and is a torch port of
kikuchipy's, so patterns land pixel for pixel where kikuchipy puts them.

In [10]:
detector = de.EBSDDetector(
    shape=(240, 240),
    pc=(0.5, 0.5, 0.6),
    sample_tilt=microscope.sample_tilt,
)
print(detector)
print(f"gnomonic bounds [x_min, x_max, y_min, y_max]: {detector.gnomonic_bounds.round(3)}")

EBSDDetector(shape=(240, 240), pc=(0.5, 0.5, 0.6), sample_tilt=70.0, tilt=0.0, azimuthal=0.0, twist=0.0, px_size=1.0)
gnomonic bounds [x_min, x_max, y_min, y_max]: [-0.833  0.833 -0.833  0.833]


An interactive widget for the geometry -- edit the tilts and pattern center and watch
the detector move relative to the sample.

In [11]:
detector.show(microscope=microscope, crystal=crystal)

Now project. Orientations are orix `Rotation`s (or plain `(N, 4)` quaternions), and a
whole stack projects in one batched call.

In [ ]:
euler_deg = np.array([
    [0.0, 0.0, 0.0],      # cube-on
    [30.0, 20.0, 10.0],
    [45.0, 54.74, 0.0],   # <111> along the beam
])
orientations = Rotation.from_euler(np.deg2rad(euler_deg))

patterns = master.project(detector, orientations)
print(f"patterns {tuple(patterns.shape)} = (n_orientations, n_rows, n_cols), {patterns.dtype}")
print(f"value range: [{patterns.min():.2f}, {patterns.max():.2f}]")

Detector row 0 is the **top** of the detector, so these draw with `origin="upper"`.

In [ ]:
from de_ebsd.viz.figures import apl, label

n = patterns.shape[0]
fig, axes = apl().subplots(1, n, figsize=(330 * n, 360))
panels = zip(np.ravel(axes), patterns.cpu().numpy(), euler_deg, strict=True)
for ax, patt, (phi1, phi, phi2) in panels:
    panel = ax.imshow(patt, cmap="gray", origin="upper")
    label(panel, f"({phi1:g}, {phi:g}, {phi2:g})")
fig

## Next steps

- `master.save("Si_master.h5")` / `de.MasterPattern.load(...)` to skip the recompute.
- `simulator.refine(npx=200)` returns a new simulator that reuses every cache still
  valid (raising `npx` does not recompute structure factors; changing `dmin` does).
- `de_ebsd.index` takes patterns back to orientations, via `SphericalIndexer` or
  `DictionaryIndexer`. Two things to know before you try it:
  - The indexers auto-derive the phase from `master.metadata["crystal"]`, but a
    *simulated* master stores that as a string, not a `Crystal` -- so pass the
    symmetry yourself: `DictionaryIndexer(master, detector, point_group=Oh)`.
  - Indexing accuracy depends on bandwidth, master resolution and the phase's
    pseudo-symmetry. Check it against known orientations on your own phase before
    trusting a map; a coarse pass that looks plausible can still be wrong.